In [1]:
%pip install ipykernel pandas numpy scikit-learn matplotlib seaborn wordcloud gensim

Note: you may need to restart the kernel to use updated packages.


# Εξόρυξη Γνώσης από Μουσικά Δεδομένα (Audio & Lyrics)

**Ομάδα:**
* Παναγιώτης Μακαρόνας (AM: sdi2300107)
* Αχιλλέας Νιανιακούδης-Κοέν (AM: sdi2300138)

In [2]:

# Main data handling libraries
import pandas as pd     # Used for dataframes handling and data manipulation
import numpy as np      # Main math library in Python

import tarfile      # Opening tarfile to read its insides
import gensim       # NLP (Natural Language Processing)

# The MLTs (Machine Learning Tools)
from sklearn.cluster import KMeans      # Great for finding clusters and grouping them
from sklearn.decomposition import PCA   # Lowering data size to help the process
from sklearn.manifold import TSNE       # Helping with organizing massive data groups for plots
from sklearn.metrics.pairwise import cosine_similarity  # Finding the similarity between two songs

# Drawing Tools
import matplotlib.pyplot as plt     # Generating plots
import seaborn as sns               # Helps in making the plots more modern
from wordcloud import WordCloud     # Image generator for most frequently used tags in a cluster


## 1. Συλλογή Δεδομένων (Data collection)
Σε αυτό το κελί φορτώνονται τα δεδομένα, εφαρμόζεται το φιλτράρισμα στα Top-5 Genres και πραγματοποιείται το intersection των αρχείων.

In [3]:

# File paths
genres_path = "data/id_genres.csv"
info_path = "data/id_information.csv"
mfcc_path = "data/id_mfcc_stats.tsv.bz2"
tags_path = "data/id_tags.csv"
lyrics_archive = "data/processed_lyrics.tar.gz"

# Load files (all are TAB-separated)
df_genres = pd.read_csv(genres_path, sep='\t')
df_info = pd.read_csv(info_path, sep='\t')
df_tags = pd.read_csv(tags_path, sep='\t')
df_mfcc = pd.read_csv(mfcc_path, sep='\t', compression='bz2')

# Each row can have multiple genres (comma-separated) — explode to one genre per row
df_genres['genre'] = df_genres['genres'].str.split(',')
df_genres = df_genres.explode('genre')
df_genres['genre'] = df_genres['genre'].str.strip()

# Top-5 genres filtering
top5 = df_genres['genre'].value_counts().head(5).index.tolist()
df_genres = df_genres[df_genres['genre'].isin(top5)]
print(f"Top-5 genres: {top5}")

# Keep one genre per song (first matching top-5 genre)
df_genres = df_genres.drop_duplicates(subset='id', keep='first')

# Intersection: keep only IDs present in genres, mfcc AND lyrics archive
genre_ids = set(df_genres['id'])
mfcc_ids = set(df_mfcc.iloc[:, 0])  # first col is the song id
common_ids = genre_ids & mfcc_ids   # songs in both genres and mfcc

# Extract lyrics only for common IDs
lyrics_dict = {}
with tarfile.open(lyrics_archive, 'r:gz') as tar:
    for member in tar:
        if not member.isfile():
            continue
        song_id = member.name.split('/')[-1].replace('.txt', '')
        if song_id in common_ids:
            f = tar.extractfile(member)
            if f:
                lyrics_dict[song_id] = f.read().decode('utf-8', errors='ignore')

# Build final clean DataFrame
df_lyrics = pd.DataFrame(list(lyrics_dict.items()), columns=['id', 'lyrics'])
final_ids = common_ids & set(df_lyrics['id'])

df = df_genres[df_genres['id'].isin(final_ids)].merge(df_lyrics, on='id')
df = df.merge(df_mfcc, left_on='id', right_on=df_mfcc.columns[0])

print(f"Final dataset: {len(df)} songs across {df['genre'].nunique()} genres")
print(df[['id', 'genre', 'lyrics']].head())


Top-5 genres: ['rock', 'pop', 'electronic', 'alternative rock', 'indie rock']
Final dataset: 56100 songs across 5 genres
                 id       genre  \
0  0009fFIM1eYThaPg         pop   
1  002Jyd0vN4HyCpqL        rock   
2  00CH4HJdxQQQbJfu  indie rock   
3  00DZ3XCAQb2FdCc6  electronic   
4  00IeldeA9ijZOL0P         pop   

                                              lyrics  
0  sunni day get nowher hide cloud sky pretend ge...  
1  goer phone freiburg say willi do quit job hitl...  
2  crimin walk room ask live way think could woul...  
3                                                     
4  yeah yeah love alway find way everi singl time...  


## 2. Εξαγωγή Χαρακτηριστικών & Embeddings (Feature Extraction)
Εδώ δημιουργούνται οι διανυσματικές αναπαραστάσεις για το κείμενο (Word2Vec/Doc2Vec) και τον ήχο (PCA στα MFCCs).

In [ ]:

# =============================================
# 2a. TEXT EMBEDDINGS  (Word2Vec → avg vector)
# =============================================

from gensim.models import Word2Vec

# Tokenize lyrics into lowercase word lists
tokenized = df['lyrics'].apply(lambda x: x.lower().split()).tolist()

# Train Word2Vec on our corpus
w2v_model = Word2Vec(sentences=tokenized, vector_size=128,
                     window=5, min_count=2, workers=4, epochs=10)
print(f"Word2Vec vocabulary size: {len(w2v_model.wv)}")

# Average all word vectors of a song into a single 128-d vector
def song_text_embedding(words, model):
    vecs = [model.wv[w] for w in words if w in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(model.vector_size)

# Build the text embeddings matrix (N_songs x 128)
text_embeddings = np.vstack(
    [song_text_embedding(tok, w2v_model) for tok in tokenized]
)
print(f"Text embeddings shape: {text_embeddings.shape}")

# =============================================
# 2b. AUDIO EMBEDDINGS  (PCA on MFCC stats)
# =============================================

# Keep only numeric MFCC columns, exclude string columns
mfcc_cols = [c for c in df.columns if c not in ['id', 'genre', 'lyrics', 'genres']]
mfcc_matrix = df[mfcc_cols].values.astype(np.float32)

# Replace NaN/Inf with 0
mfcc_matrix = np.nan_to_num(mfcc_matrix, nan=0.0, posinf=0.0, neginf=0.0)

# PCA: reduce ~140 dims keeping 95% of variance
pca = PCA(n_components=0.95, random_state=42)
audio_embeddings = pca.fit_transform(mfcc_matrix)
print(f"Audio PCA: {mfcc_matrix.shape[1]} dims → {audio_embeddings.shape[1]} dims "
      f"({pca.n_components_} components, 95% variance)")

# Store embeddings in the DataFrame
df['text_emb'] = list(text_embeddings)
df['audio_emb'] = list(audio_embeddings)
print("Embeddings stored in DataFrame ✓")


Word2Vec vocabulary size: 30177
Text embeddings shape: (56100, 128)
Audio PCA: 104 dims → 58 dims (58 components, 95% variance)
Embeddings stored in DataFrame ✓


## 3. Οπτικοποίηση και Ανάλυση (Exploratory Data Analysis - EDA)
Ανάλυση των tags, word clouds, 2D projections (PCA/t-SNE) και Cosine Similarity.

In [ ]:

# =============================================
# 3a. WORD CLOUDS — Two most different genres
# =============================================

# Merge tags into main DataFrame
df_tags['id'] = df_tags['id'].astype(str).str.strip()
df_with_tags = df.merge(df_tags, on='id', how='left')

# Pick first and last genre alphabetically (most different pair)
genre_list = sorted(df['genre'].unique())
genre_a, genre_b = genre_list[0], genre_list[-1]
print(f"Word clouds for: '{genre_a}' vs '{genre_b}'")

# One word cloud per genre, side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, genre in zip(axes, [genre_a, genre_b]):
    genre_tags = df_with_tags[df_with_tags['genre'] == genre]['tags'].dropna()
    all_tags_text = ' '.join(genre_tags.astype(str))
    
    wc = WordCloud(width=800, height=400, background_color='white',
                   colormap='viridis', max_words=100).generate(all_tags_text)
    
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(f"Word Cloud — {genre}", fontsize=14, fontweight='bold')
    ax.axis('off')

plt.suptitle("Tag Word Clouds: Two Most Different Genres", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# =============================================
# 3b. BAR CHART — Top 10 most frequent tags
# =============================================

# Split comma-separated tags, flatten, and count
all_tags = df_with_tags['tags'].dropna().astype(str)
tag_series = all_tags.str.split(',').explode().str.strip()
tag_counts = tag_series.value_counts().head(10)

# Horizontal bar chart
plt.figure(figsize=(10, 6))
sns.barplot(x=tag_counts.values, y=tag_counts.index, palette='viridis')
plt.xlabel("Frequency", fontsize=12)
plt.ylabel("Tag", fontsize=12)
plt.title("Top 10 Most Frequent Tags Across All Songs", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# =============================================
# 3c. 2D SCATTER PLOTS — Text & Audio Embeddings
# =============================================

# t-SNE to reduce embeddings to 2D for visualization
tsne_text = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
text_2d = tsne_text.fit_transform(text_embeddings)

tsne_audio = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
audio_2d = tsne_audio.fit_transform(audio_embeddings)

# Side-by-side scatter plots colored by genre
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, data_2d, title in zip(axes, [text_2d, audio_2d], ["Text Embeddings (t-SNE)", "Audio Embeddings (t-SNE)"]):
    for genre in df['genre'].unique():
        mask = df['genre'] == genre
        ax.scatter(data_2d[mask, 0], data_2d[mask, 1], label=genre, alpha=0.5, s=15)
    
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel("Dimension 1")
    ax.set_ylabel("Dimension 2")
    ax.legend(fontsize=9, markerscale=2)

plt.suptitle("2D Projections of Embeddings — Colored by Genre", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# =============================================
# 3d. COSINE SIMILARITY — 5 nearest songs
# =============================================

# Query song: find the 5 most similar songs by text and by audio
query_idx = 0
query_song = df.iloc[query_idx]
print(f"\n{'='*60}")
print(f"Query song: ID={query_song['id']}, Genre={query_song['genre']}")
print(f"{'='*60}")

# Text-based: cosine similarity on text embeddings
text_sim = cosine_similarity([text_embeddings[query_idx]], text_embeddings)[0]
text_sim[query_idx] = -1                                      # Exclude self
top5_text = np.argsort(text_sim)[-5:][::-1]

print("\n--- Top 5 Similar Songs (by TEXT / Lyrics) ---")
for rank, idx in enumerate(top5_text, 1):
    print(f"  {rank}. ID={df.iloc[idx]['id']}  Genre={df.iloc[idx]['genre']}  "
          f"Similarity={text_sim[idx]:.4f}")

# Audio-based: cosine similarity on audio embeddings
audio_sim = cosine_similarity([audio_embeddings[query_idx]], audio_embeddings)[0]
audio_sim[query_idx] = -1                                     # Exclude self
top5_audio = np.argsort(audio_sim)[-5:][::-1]

print("\n--- Top 5 Similar Songs (by AUDIO / MFCCs) ---")
for rank, idx in enumerate(top5_audio, 1):
    print(f"  {rank}. ID={df.iloc[idx]['id']}  Genre={df.iloc[idx]['genre']}  "
          f"Similarity={audio_sim[idx]:.4f}")

# If Text and Audio return different genres, the two modalities capture different aspects of similarity
print("\n[Observation] Compare the genres above: if Text and Audio return different genres,")
print("it means lyrics and sound encode different aspects of music similarity.")


## ΜΕΡΟΣ Β: Κατηγοριοποίηση & Clustering